In [0]:
dbutils.library.restartPython()

In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from pyspark.sql import functions as F
from lh import common, silver

cfg = common.load_config("dev")
silver.ensure_tables(spark, cfg)
cat = cfg["catalog"]
spec = {"key": ["id"], "order": "last_modified", "mode": "latest",
        "columns": {"id": "string", "city": "city", "last_modified": "timestamp"}}
rows = [("K1", "x", "2026-10-15 09:00:00", "2026-10-20"),      # 5 days late: fine
        ("K2", "x", "2026-10-10 09:00:00", "2026-10-20"),      # 10 days late: too_late
        ("K3", "x", "2026-09-01 09:00:00", "2026-09-30"),      # history file: exempt
        ("K4", "x", "2026-10-13 09:00:00", "2026-10-20")]      # exactly 7 days: fine
sample = (spark.createDataFrame(rows, "id string, city string, last_modified string, bd string")
               .withColumn("_business_date", F.to_date("bd")).drop("bd"))
got = {r["id"]: r["_rule"] for r in silver.check(silver.apply(sample, spec), spec, cfg).collect()}
print(got, "PASS" if got == {"K1": None, "K2": "too_late", "K3": None, "K4": None} else "FAIL")

In [0]:
def add(rows):                                        # append rows to a small bronze test table
    (spark.createDataFrame(rows, "id string, city string, last_modified string, ts string, bd string")
          .withColumn("_ingest_ts", F.to_timestamp("ts")).withColumn("_business_date", F.to_date("bd"))
          .withColumn("_source_file", F.lit("t09.csv")).drop("ts", "bd")
          .write.mode("append").saveAsTable(f"{cat}.bronze.t09_test"))

def clean_up():
    spark.sql(f"DROP TABLE IF EXISTS {cat}.bronze.t09_test")
    spark.sql(f"DROP TABLE IF EXISTS {cat}.silver.t09_test")
    spark.sql(f"DELETE FROM {cat}.ops.dq_results WHERE table_name = 't09_test'")
    spark.sql(f"DELETE FROM {cat}.ops.silver_state WHERE table_name = 't09_test'")

clean_up()
steps = []
add([("K1", "BLR", "2026-09-10 10:00:00", "2026-09-11 00:30:00", "2026-09-10")])
steps.append(silver.load_latest(spark, cfg, "t09_test", spec))        # first version
add([("K1", "Bombay", "2026-09-09 09:00:00", "2026-09-12 00:30:00", "2026-09-11")])
steps.append(silver.load_latest(spark, cfg, "t09_test", spec))        # older version arrives late: ignored
add([("K1", "Madras", "2026-09-12 11:00:00", "2026-09-13 00:30:00", "2026-09-12"),
     ("K2", "hyd", "2026-09-12 12:00:00", "2026-09-13 00:30:00", "2026-09-12")])
steps.append(silver.load_latest(spark, cfg, "t09_test", spec))        # newer version updates, K2 is new
city = {r["id"]: r["city"] for r in spark.table(f"{cat}.silver.t09_test").collect()}
clean_up()
got = [(s["rows"], s["inserted"], s["updated"]) for s in steps]
print(got, city)
print("PASS" if got == [(1, 1, 0), (1, 0, 0), (2, 1, 1)] and city == {"K1": "Chennai", "K2": "Hyderabad"} else "FAIL")

In [0]:
tables = silver.load_silver()
want = {"appointments": 2192, "admissions": 269, "treatments": 506,
        "pharmacy_sales": 1644, "invoices": 2169, "payments": 1977}
start = time.time()
second = quarantined = 0
for run in (1, 2):                                   # the second run must find nothing new
    for table in want:
        r = silver.load_latest(spark, cfg, table, tables[table])
        print(f"run {run} {table:16} {r}")
        quarantined += r["quarantined"]
        if run == 2:
            second += r["rows"] + r["inserted"] + r["updated"]
got = {t: spark.table(f"{cat}.silver.{t}").count() for t in want}
print(got, f"in {time.time() - start:.0f}s")
print("PASS" if got == want and second == 0 and quarantined == 0 else "FAIL")